# 01 · Smoke test (GPU)

Checks the teacher scoring on 50 training rows before any real labeling:

1. **vLLM** scores the rows with P2 zero-shot (`p2-zs-v1`), reading the top-20 logprobs of the
   first generated token.
2. **Hugging Face** scores the same rows with an exact softmax at the last prompt position.
3. **Throughput:** vLLM scores a larger random sample to estimate full-labeling hours.

**Accept when:** no NaN or infinite probabilities, mean coverage ≥ 0.9, and mean absolute
difference between vLLM and Hugging Face probabilities < 0.02. The last cell raises if a check fails.

**Session settings:** Accelerator **GPU T4 x2**, Internet **On**. vLLM uses GPU 0, Hugging Face
uses GPU 1.

**Inputs:** Add Input → Your Work → Notebooks → **00 prepare data** (its `wiki/` and
`qwen3-4b-instruct-2507/` output).

**If vLLM fails to install or load:** set `RUN_VLLM = False` in the next cell and run all again.
Hugging Face then scores everything and its throughput is recorded instead (plan, Step 2).

Results are written to `smoke_results.json` in the output.

In [ ]:
REPO_URL = "https://github.com/Nishaant-Soni/Distillate.git"
REPO_REF = "Nishaant/Teacher"     # use a commit hash once this notebook's code is on main
VLLM_VERSION = "0.31.0"           # supports T4 (compute capability 7.5)
RUN_VLLM = True
SMOKE_ROWS = 50                   # 25 toxic + 25 other rows, for the vLLM/HF comparison
THROUGHPUT_ROWS = 2000            # random rows for the vLLM throughput estimate
HF_THROUGHPUT_ROWS = 200          # used only when RUN_VLLM is False

In [ ]:
# Install vLLM before anything imports torch: vLLM pins its own torch version.
import subprocess
import sys

if RUN_VLLM:
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", f"vllm=={VLLM_VERSION}"],
                   check=True)

In [ ]:
from pathlib import Path

REPO_DIR = Path("/tmp/Distillate")   # outside /kaggle/working, so the repo is not saved as output
if not REPO_DIR.exists():
    subprocess.run(["git", "clone", "--quiet", REPO_URL, str(REPO_DIR)], check=True)
subprocess.run(["git", "-C", str(REPO_DIR), "fetch", "--quiet", "origin"], check=True)
subprocess.run(["git", "-C", str(REPO_DIR), "checkout", "--quiet", REPO_REF], check=True)
CODE_COMMIT = subprocess.check_output(
    ["git", "-C", str(REPO_DIR), "rev-parse", "HEAD"], text=True).strip()
sys.path.insert(0, str(REPO_DIR / "Model-Pipeline"))
print("code_commit:", CODE_COMMIT)

In [ ]:
import json
import time

import numpy as np
import pandas as pd

from src.teacher.config import LABELS, load_config, model_version
from src.teacher.data import find_file

cfg = load_config()
SEED = cfg["seed"]
MIN_COVERAGE = cfg["scoring"]["min_coverage"]
MAX_RETRIES = cfg["scoring"]["max_retries"]

TRAIN_PATH = find_file("/kaggle/input", cfg["data"]["files"]["train"])
MODEL_DIR = find_file("/kaggle/input", "weights_manifest.json").parent
weights_manifest = json.loads((MODEL_DIR / "weights_manifest.json").read_text())
assert weights_manifest["revision"] == cfg["model"]["revision"], weights_manifest["revision"]
print("train:", TRAIN_PATH)
print("model:", MODEL_DIR, "@", weights_manifest["revision"])

results = {
    "code_commit": CODE_COMMIT,
    "model_version": model_version(cfg),
    "prompt_version": "p2-zs-v1",
}

In [ ]:
# 25 random toxic rows and 25 random other rows, so the probabilities cover the whole range.
train = pd.read_parquet(TRAIN_PATH)
toxic = train[train["toxicity"] == 1].sample(SMOKE_ROWS // 2, random_state=SEED)
other = train.drop(toxic.index).sample(SMOKE_ROWS - len(toxic), random_state=SEED)
smoke = pd.concat([toxic, other]).reset_index(drop=True)
ids, texts = smoke["id"].tolist(), smoke["text"].tolist()
print(len(smoke), "rows;", int(smoke["toxicity"].sum()), "toxic")

In [ ]:
from transformers import AutoTokenizer

from src.teacher.prompts import PromptBuilder
from src.teacher.scoring import answer_token_ids, score_texts

tokenizer = AutoTokenizer.from_pretrained(MODEL_DIR)
builder = PromptBuilder.from_config(cfg, tokenizer, protocol="p2", shots="zs")
yes_ids, no_ids = answer_token_ids(tokenizer)
print("prompt_version:", builder.prompt_version, "| max prompt tokens:", builder.max_prompt_tokens)
print("YES_IDS", sorted(yes_ids), "NO_IDS", sorted(no_ids))

## 1. vLLM (GPU 0)

In [ ]:
if RUN_VLLM:
    import vllm
    from vllm import LLM

    from src.teacher.scoring import vllm_score_fn

    start = time.perf_counter()
    llm = LLM(
        model=str(MODEL_DIR),
        dtype=cfg["model"]["dtype"],
        max_model_len=cfg["model"]["max_model_len"],
        seed=SEED,
        gpu_memory_utilization=0.90,
        enable_prefix_caching=True,   # the system message and comment are shared by six probes
        generation_config="vllm",     # ignore the model's sampling defaults; we set our own
    )
    results["vllm_version"] = vllm.__version__
    results["vllm_load_s"] = round(time.perf_counter() - start, 1)
    score_vllm = vllm_score_fn(llm, yes_ids, no_ids, logprobs=cfg["scoring"]["logprobs"])
    print("vLLM", vllm.__version__, "loaded in", results["vllm_load_s"], "s")

In [ ]:
if RUN_VLLM:
    start = time.perf_counter()
    vdf, vinfo = score_texts(ids, texts, builder, score_vllm, MIN_COVERAGE, MAX_RETRIES)
    results["vllm_smoke_s"] = round(time.perf_counter() - start, 2)
    results["vllm_smoke_info"] = vinfo
    print(vinfo)
    display(vdf.head())

In [ ]:
# Throughput on a larger random sample. The first call above already warmed vLLM up.
if RUN_VLLM:
    sample = train.sample(THROUGHPUT_ROWS, random_state=SEED + 1)
    start = time.perf_counter()
    tdf, tinfo = score_texts(sample["id"].tolist(), sample["text"].tolist(), builder,
                             score_vllm, MIN_COVERAGE, MAX_RETRIES)
    elapsed = time.perf_counter() - start
    results["throughput"] = {
        "backend": "vllm",
        "rows": THROUGHPUT_ROWS,
        "seconds": round(elapsed, 1),
        "rows_per_s": round(THROUGHPUT_ROWS / elapsed, 2),
        "prompts_per_s": round(tinfo["prompts_scored"] / elapsed, 1),
        "failed_rows": tinfo["failed_rows"],
        "truncated_rows": tinfo["truncated_rows"],
    }
    print(results["throughput"])

## 2. Hugging Face exact probabilities (GPU 1)

In [ ]:
import torch
import transformers
from transformers import AutoModelForCausalLM

from src.teacher.scoring import hf_score_fn

HF_DEVICE = 1 if torch.cuda.device_count() > 1 else 0
start = time.perf_counter()
hf_model = AutoModelForCausalLM.from_pretrained(MODEL_DIR, dtype=torch.float16)
hf_model = hf_model.to(f"cuda:{HF_DEVICE}").eval()
results["hf_load_s"] = round(time.perf_counter() - start, 1)
torch.cuda.reset_peak_memory_stats(HF_DEVICE)
score_hf = hf_score_fn(hf_model, tokenizer, yes_ids, no_ids, batch_size=8)

start = time.perf_counter()
hdf, hinfo = score_texts(ids, texts, builder, score_hf, MIN_COVERAGE, MAX_RETRIES)
results["hf_smoke_s"] = round(time.perf_counter() - start, 2)
results["hf_smoke_info"] = hinfo
results["hf_peak_memory_gb"] = round(torch.cuda.max_memory_allocated(HF_DEVICE) / 1e9, 2)
results["torch_version"] = torch.__version__
results["transformers_version"] = transformers.__version__
print(hinfo, "| peak memory", results["hf_peak_memory_gb"], "GB")

In [ ]:
# Only when vLLM is not used: Hugging Face throughput for the labeling estimate.
if not RUN_VLLM:
    sample = train.sample(HF_THROUGHPUT_ROWS, random_state=SEED + 1)
    start = time.perf_counter()
    tdf, tinfo = score_texts(sample["id"].tolist(), sample["text"].tolist(), builder,
                             score_hf, MIN_COVERAGE, MAX_RETRIES)
    elapsed = time.perf_counter() - start
    results["throughput"] = {
        "backend": "hf",
        "rows": HF_THROUGHPUT_ROWS,
        "seconds": round(elapsed, 1),
        "rows_per_s": round(HF_THROUGHPUT_ROWS / elapsed, 2),
        "prompts_per_s": round(tinfo["prompts_scored"] / elapsed, 1),
        "failed_rows": tinfo["failed_rows"],
        "truncated_rows": tinfo["truncated_rows"],
    }
    print(results["throughput"])

## 3. Checks and results

In [ ]:
P = [f"p_{label}" for label in LABELS]
C = [f"cov_{label}" for label in LABELS]
main = vdf if RUN_VLLM else hdf          # the backend labeling will use
ok = main["status"] == "ok"
finite = bool(np.isfinite(main.loc[ok, P].to_numpy()).all())
checks = {
    "all_rows_ok_no_nan_or_inf": finite and bool(ok.all()),
    "mean_coverage_ge_0.9": float(main[C].to_numpy().mean()) >= 0.9,
}
results["mean_coverage"] = round(float(main[C].to_numpy().mean()), 4)
results["min_coverage_seen"] = round(float(main[C].to_numpy().min()), 4)

if RUN_VLLM:
    both = (vdf["status"] == "ok") & (hdf["status"] == "ok")
    diff = np.abs(vdf.loc[both, P].to_numpy() - hdf.loc[both, P].to_numpy())
    # Probability mass the top-20 list missed: exact coverage minus top-20 coverage.
    missed = hdf.loc[both, C].to_numpy() - vdf.loc[both, C].to_numpy()
    results["vllm_vs_hf"] = {
        "rows_compared": int(both.sum()),
        "mean_abs_diff": round(float(diff.mean()), 5),
        "max_abs_diff": round(float(diff.max()), 5),
        "mean_abs_diff_per_label": {label: round(float(d), 5)
                                    for label, d in zip(LABELS, diff.mean(axis=0), strict=True)},
        "mean_missed_mass": round(float(missed.mean()), 6),
    }
    checks["vllm_hf_mean_abs_diff_lt_0.02"] = results["vllm_vs_hf"]["mean_abs_diff"] < 0.02

tp = results["throughput"]
n_train = cfg["data"]["expected_rows"]["train"]
tp["est_hours_train_one_gpu"] = round(n_train / tp["rows_per_s"] / 3600, 2)
tp["est_hours_train_two_gpus"] = round(tp["est_hours_train_one_gpu"] / 2, 2)
results["checks"] = checks
results["gpu"] = subprocess.check_output(
    ["nvidia-smi", "--query-gpu=index,name,memory.used,memory.total", "--format=csv,noheader"],
    text=True).strip().splitlines()

Path("/kaggle/working/smoke_results.json").write_text(json.dumps(results, indent=2))
print(json.dumps(results, indent=2))

In [ ]:
# Side by side, for a visual check (no comment text shown).
if RUN_VLLM:
    compare = pd.DataFrame({"id": ids, "toxicity_gold": smoke["toxicity"]})
    for label in ("toxicity", "insult", "threat"):
        compare[f"vllm_{label}"] = vdf[f"p_{label}"].round(3)
        compare[f"hf_{label}"] = hdf[f"p_{label}"].round(3)
    display(compare.head(15))

In [ ]:
failed = [name for name, passed in checks.items() if not passed]
assert not failed, f"smoke test failed: {failed}"
print("Smoke test passed:", ", ".join(checks))